In [ ]:
#| default_exp exhash

In [ ]:
#| export
from fastcore.utils import *
from dialoghelper.core import *
from exhash import *

In [ ]:
msg = await read_msg()
id_ = msg.id
cts = msg.content

In [ ]:
#| export
async def msg_lnhashview(id:str):
    "Show lnhash-addressed lines of a message"
    msg = await read_msgid(id=id)
    return '\n'.join(lnhashview(msg['content']))

In [ ]:
#| export
async def msg_exhash(id:str, cmds:list):
    """Verified line-addressed editor. Apply commands to msg `id` contents, return lnhashview(result).
    **NB**: *all* exhash commands *must* start with an address.
    The *only* allowed addresses are a single lnhash, or a pair separated by `,`. (I.e no `%`, `.`, etc.)
    **NB**: hashes are checked before each command is run. So be sure to have commands go last->first order to avoid changing earlier lines.

    Commands are like `ex`, but use lnhash addresses instead of bare line numbers: ``lineno|hash|cmd`` where hash is a 4-char hex content hash.
    Use `msg_lnhashview(text)` to get addresses before first use.

    Addressing:
      Single:   ``12|a3f2|cmd``
      Range:    ``12|a3f2|,15|b1c3|cmd``
      Special:  ``0|0000|`` targets before line 1 (only with a or i)

    Commands:
      s/pat/rep/[flags]  Substitute (regex). Flags: g=all, i=case-insensitive
      d                  Delete line(s)
      a                  Append text after line
      i                  Insert text before line
      c                  Change/replace line(s)
      j                  Join with next line; with range, joins all
      m dest             Move line(s) after dest address
      t dest             Copy line(s) after dest address
      >[n]               Indent n levels (default 1, 4 spaces each)
      <[n]               Dedent n levels (default 1)
      sort               Sort lines alphabetically
      p                  Print (include in output without changing)
      g/pat/cmd          Global: run cmd on matching lines
      g!/pat/cmd         Inverted global (also v/pat/cmd)

    `cmds` is a required list of command strings. For `a`/`i`/`c`, include the text block in the same command string after a newline."""
    msg = await read_msgid(id=id)
    txt = msg['content']
    res = exhash(txt, cmds)
    res = '\n'.join(res['lines'])
    upres = await update_msg(id=id, content=res)
    assert upres.startswith('_'), f"Message update failed: {upres}"
    return '\n'.join(lnhashview(res))

In [ ]:
#| export
def file_lnhashview(path:str):
    "Show lnhash-addressed lines of a file"
    return '\n'.join(lnhashview(Path(path).read_text()))

In [ ]:
#| export
def file_exhash(path:str, cmds:list[str]):
    """Verified line-addressed editor for files. Apply commands to file at `path`, return lnhashview(result).
    See `doc(msg_lnhashview)` for details"""
    txt = Path(path).read_text()
    res = '\n'.join(exhash(txt, cmds)['lines'])
    Path(path).write_text(res)
    return '\n'.join(lnhashview(res))

In [ ]:
tp = Path('tmp.txt')
tp.write_text(cts)

313

Try indenting the first 3 lines of this file with &`[file_lnhashview,file_exhash]`.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->


<details class='tool-usage-details'>
<summary>file_lnhashview(path='tmp.txt')</summary>

```json
{
  "id": "toolu_01T723e8w3tcYjgdEwNopV18",
  "call": {
    "function": "file_lnhashview",
    "arguments": {
      "path": "tmp.txt"
    }
  },
  "result": "1|da2a|  The quick brown fox jumps over the lazy dog.\n2|32e4|  Jackdaws love my big sphinx of quartz.\n3|04e4|  Pack my box with five dozen liquor jugs.\n4|025f|  How vexingly quick daft zebras jump!\n5|c720|  The five boxing wizards jump quickly.\n6|0666|  Sphinx of black quartz, judge my vow.\n7|d61c|  Two driven jocks help fax my big quiz.\n8|bc1b|  Bright vixens jump; dozy fowl quack."
}
```

</details>

<details class='tool-usage-details'>
<summary>file_exhash(path='tmp.txt', cmds=['1|da2a|,3|04e4|>'])</summary>

```json
{
  "id": "toolu_01QdybzYCodQBHB5WKMo7Z92",
  "call": {
    "function": "file_exhash",
    "arguments": {
      "path": "tmp.txt",
      "cmds": "['1|da2a|,3|04e4|>']"
    }
  },
  "result": "1|dfab|      The quick brown fox jumps over the lazy dog.\n2|e110|      Jackdaws love my big sphinx of quartz.\n3|a4ce|      Pack my box with five dozen liquor jugs.\n4|025f|  How vexingly quick daft zebras jump!\n5|c720|  The five boxing wizards jump quickly.\n6|0666|  Sphinx of black quartz, judge my vow.\n7|d61c|  Two driven jocks help fax my big quiz.\n8|bc1b|  Bright vixens jump; dozy fowl quack."
}
```

</details>

Done — first 3 lines are now indented. You can see the hashes changed for those lines (`da2a`→`dfab`, etc.) since the content now includes leading spaces.


In [ ]:
print(tp.read_text())

    The quick brown fox jumps over the lazy dog.
    Jackdaws love my big sphinx of quartz.
    Pack my box with five dozen liquor jugs.
How vexingly quick daft zebras jump!
The five boxing wizards jump quickly.
Sphinx of black quartz, judge my vow.
Two driven jocks help fax my big quiz.
Bright vixens jump; dozy fowl quack.


In [ ]:
tp.unlink()